# Redes neuronales cuánticas con y sin qang

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_qml_es.ipynb)

Este cuaderno usa la librería publicada (`pip install qang`, módulo `qang.qml`). Una QNN que conserva el peso de Hamming mantiene los datos en un sector, y el **filtro qg** puede quedarse solo con los disparos que siguieron en ese sector. Cada resultado se muestra **con qang** (lectura filtrada) y **sin qang** (lectura cruda), con su diferencia.

Qué esperar (RESEARCH_NOTES §75–§80): no hay ventaja cuántica (estos modelos son simulables clásicamente); entrenada en simulador y ejecutada con relajación T1, qang recupera exactamente la precisión sin ruido; entrenada con el ruido calibrado, la versión sin qang la alcanza; el desfase y el T1 desigual no los corrige el filtro.

## 1. Instalación

In [ ]:
!pip install -q -U "qang>=0.6.0" scikit-learn
import qang
print('qang', qang.__version__)

## 2. Datos: cáncer de mama, 4 componentes PCA escaladas a [-1, 1]

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from qang.qml import WeightQNN, compare_qang, kept_fraction

d = load_breast_cancer()
rng = np.random.default_rng(0)
idx = np.concatenate([rng.choice(np.where(d.target == c)[0], 100, replace=False) for c in (0, 1)])
Xtr, Xte, ytr, yte = train_test_split(d.data[idx], d.target[idx], test_size=0.3, stratify=d.target[idx], random_state=0)
sc = StandardScaler().fit(Xtr); pca = PCA(4, random_state=0).fit(sc.transform(Xtr))
Ztr, Zte = pca.transform(sc.transform(Xtr)), pca.transform(sc.transform(Xte))
lo, hi = Ztr.min(0), Ztr.max(0)
scale = lambda Z: np.clip(2 * (Z - lo) / (hi - lo) - 1, -1, 1)
Xtr, Xte = scale(Ztr), scale(Zte)
print(Xtr.shape, Xte.shape)

def show(rows, cols=('lectura', 'con qang', 'sin qang', 'diferencia')):
    print(f"{cols[0]:<34}{cols[1]:>11}{cols[2]:>13}{cols[3]:>12}")
    for name, q, r in rows:
        print(f"{name:<34}{q:>11.3f}{r:>13.3f}{q - r:>+12.3f}")

## 3. QNN de peso 1, entrenada sin ruido, ejecutada con T1 (gamma = 0.08)

Con qang la lectura es exactamente la sin ruido (F1); el precio son los disparos descartados, conocidos de antemano: (1 - gamma)^(peso x profundidad).

In [ ]:
GAMMA = 0.08
m1 = WeightQNN(n_qubits=5, weight=1).fit(Xtr, ytr, epochs=120, seed=0)
exact1 = m1.score(Xte, yte)
show([("weight 1, T1, trained clean", m1.score(Xte, yte, gamma=GAMMA, qang=True), m1.score(Xte, yte, gamma=GAMMA, qang=False))])
print(f"noiseless accuracy {exact1:.3f}; kept fraction with qang {kept_fraction(GAMMA, 1, m1.depth):.3f}")

## 4. QNN de peso 2: el efecto de qang crece con el peso

Un estado de peso 2 decae el doble de rápido: sin qang la lectura está más sesgada; con qang sigue siendo exacta.

In [ ]:
m2 = WeightQNN(n_qubits=5, weight=2).fit(Xtr, ytr, epochs=120, seed=0)
exact2 = m2.score(Xte, yte)
show([("weight 2, T1, trained clean", m2.score(Xte, yte, gamma=GAMMA, qang=True), m2.score(Xte, yte, gamma=GAMMA, qang=False))])
print(f"noiseless accuracy {exact2:.3f}; kept fraction with qang {kept_fraction(GAMMA, 2, m2.depth):.3f}")

## 5. Entrenamiento con ruido, con y sin qang

Si se conoce el ruido del equipo y se entrena con él, la versión sin qang aprende el decaimiento y la alcanza (§79–§80). Tarda un par de minutos.

In [ ]:
r = compare_qang(WeightQNN(5, 2), Xtr, ytr, Xte, yte, gamma=GAMMA, epochs=120, seed=0, noise_aware=True)
show([("weight 2, T1, trained under noise", r["with qang"], r["without qang"])])

## 6. Lo que el filtro no corrige: T1 distinto en cada qubit, y desfase

In [ ]:
unequal = [0.04, 0.06, 0.08, 0.10, 0.12]
show([
    ("weight 1, unequal T1", m1.score(Xte, yte, gamma=unequal, qang=True), m1.score(Xte, yte, gamma=unequal, qang=False)),
    ("weight 1, T1 + dephasing 0.03", m1.score(Xte, yte, gamma=GAMMA, dephasing=0.03, qang=True), m1.score(Xte, yte, gamma=GAMMA, dephasing=0.03, qang=False)),
    ("weight 2, unequal T1", m2.score(Xte, yte, gamma=unequal, qang=True), m2.score(Xte, yte, gamma=unequal, qang=False)),
    ("weight 2, T1 + dephasing 0.03", m2.score(Xte, yte, gamma=GAMMA, dephasing=0.03, qang=True), m2.score(Xte, yte, gamma=GAMMA, dephasing=0.03, qang=False)),
])
print(f"noiseless: weight 1 {exact1:.3f}, weight 2 {exact2:.3f}")

## 7. Disparos finitos (200 por entrada)

In [ ]:
show([
    ("weight 1, T1, 200 shots", m1.score(Xte, yte, gamma=GAMMA, qang=True, shots=200, seed=1), m1.score(Xte, yte, gamma=GAMMA, qang=False, shots=200, seed=1)),
    ("weight 2, T1, 200 shots", m2.score(Xte, yte, gamma=GAMMA, qang=True, shots=200, seed=1), m2.score(Xte, yte, gamma=GAMMA, qang=False, shots=200, seed=1)),
])

## Resumen

* **Entrenada en simulador, ejecutada con T1:** qang es decisivo, exacto con T1 igual, y más cuanto mayor es el peso.
* **Entrenada con el ruido calibrado:** con y sin qang quedan a menos de una muestra de prueba.
* **Costo:** una fracción conocida de disparos, 1 - (1 - gamma)^(peso x profundidad).
* **Límites:** el T1 desigual y el desfase no se corrigen; no hay ventaja cuántica.

Detalles, predicciones pre-registradas y fallas: `RESEARCH_NOTES.md` §75–§80, scripts en `examples/qnn_*_qg.py`.